#### Simple sequential LLM workflow

START --> LLM QA --> END 

In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace # add HF_TOKEN in .env file
from typing import TypedDict
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

True

In [3]:
# create the model
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation"
)

model = ChatHuggingFace(llm=llm)
model.invoke("what is the capital of India")

c:\Users\itpl59\Desktop\ITPL\Projects\python-examples\langgraph\lang-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AIMessage(content='The capital of India is **New Delhi**. It is located in the National Capital Territory (NCT) of Delhi and has been the capital of India since 1911.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 16, 'total_tokens': 53}, 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a064f2-3930-72c1-aee5-a9abd99675ab-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 37, 'total_tokens': 53})

In [4]:
# 2. Define the state 
class LLMState(TypedDict):
    question: str 
    answer: str 

In [5]:
def llm_qa(state: LLMState) -> LLMState:

    question = state['question']
    prompt = f"Answer the following {question}"

    content = model.invoke(prompt).content 
    state['answer'] = content
    return state 


In [6]:
graph = StateGraph(LLMState)

graph.add_node('llm_qa',llm_qa)

graph.add_edge(START,'llm_qa')
graph.add_edge('llm_qa',END)

workflow = graph.compile()

In [7]:
initial_state = {'question':'What is the capital of India'}
final_state = workflow.invoke(initial_state)
print(final_state)


{'question': 'What is the capital of India', 'answer': 'The capital of India is New Delhi.'}
